# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to use the [mlcroissant](https://github.com/mlcommons/croissant) library to explore the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) Croissant dataset package: **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells**.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Inspect the dataset metadata (access properties, not index)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Each entity (record set, field, column, etc.) is referenced by its `@id`.

Let's discover the available record sets and their main fields.

In [ ]:
# List all record set @ids
record_set_objs = dataset.metadata.record_sets

print("Available record sets and their main fields (using @ids):\n")
rs_ids = []
for rs in record_set_objs:
    print(f"RecordSet: {rs.id}")
    rs_ids.append(rs.id)
    print("  Name  :", rs.name)
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.id}: {field.name}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

For demonstration, we'll extract data from the first available record set.

In [ ]:
# Extract data from each record set by their @id
dataframes = {}
for rs_id in rs_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for RecordSet @id '{rs_id}' with shape {df.shape}")
        print(f"Fields (@id): {list(df.columns)}\n")

# Let's pick the first record set for demonstration
if rs_ids:
    records_set_id = rs_ids[0]
    print(f"Using record set: {records_set_id}")
    print(dataframes[records_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For demonstration, let's select a numeric field from the chosen record set.

In [ ]:
# List the columns and pick a numeric one (by inspection, e.g., MW - Molecular Weight, pI, or normalized abundance if present)
df = dataframes[records_set_id]
print("Fields (columns) in this record set:")
print(list(df.columns))

# For this protein dataset, likely numeric fields (by @id) might include MW, pi, or normalized abundance.
# Let's auto-detect a numeric column:
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col]) or np.issubdtype(df[col].dtype, np.number)]
if not numeric_fields:
    # Try to convert columns that look numeric
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
        except:
            pass
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

print("Numeric fields detected:", numeric_fields)

# Pick the first numeric field for demonstration
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field: {numeric_field_id}")
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > mean ({threshold:.2f}):")
    print(filtered_df[[numeric_field_id]].head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by another field if available
    # Try to find a string/categorical field
    cat_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field_id]
    if cat_fields:
        group_field = cat_fields[0]
        print(f"\nGrouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Simple histogram of the numeric field
if numeric_fields:
    plt.figure(figsize=(8,4))
    df[numeric_field_id].hist(bins=30, color='skyblue')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.title(f"Distribution of {numeric_field_id} in record set {records_set_id}")
    plt.show()

# If a categorical field is available, boxplot by group
if numeric_fields and cat_fields:
    plt.figure(figsize=(10,5))
    df.boxplot(column=numeric_field_id, by=group_field)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.suptitle("")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and process the **FAIR² Mass Spectrometry** protein dataset using the `mlcroissant` library. We accessed the dataset schema, inspected record sets and fields (by `@id`), filtered and normalized data, and visualized key distributions. This workflow can be generalized to other Croissant datasets for efficient and standardized data exploration.

_Notebook generated using the mlcroissant template and FAIR² Croissant schema (https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)._